# Assamese model: full-corpus pretraining (Colab)

Runs the full training corpus on a Colab GPU, checkpointing to Google Drive so a session disconnect never loses progress. Assumes the `phase-2` branch of this repository has already been pushed to GitHub, and that the local peak_lr/min_lr sweep (`smoke_test.py`) has already run and picked real values in `configs/model_config.json`.

Corpus splits are not in the git repo (`data/splits/` is gitignored, Drive-only). This notebook reads them directly from Drive at `individual-project-abhinavborah/assamese/splits/{train,val}.txt` after the mount cell runs -- no separate download step needed, as long as that folder is present under the same Google account's My Drive.

To resume after a disconnect: reconnect the runtime and re-run every cell from the top. The tokenized data and the checkpoint both persist on Drive, so the training loop below picks up automatically from the last saved step instead of starting over.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-abhinavborah'
REPO_DIR = '/content/individual-project-abhinavborah'

if not os.path.exists(REPO_DIR):
    !git clone -b phase-2 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git checkout phase-2 && git pull

In [ ]:
!pip install -q sentencepiece

In [ ]:
import copy
import json
import sys
import time
from pathlib import Path

import numpy as np
import sentencepiece as spm
import torch

LANGUAGE = 'assamese'
REPO_ROOT = Path(REPO_DIR) / LANGUAGE

sys.path.insert(0, str(REPO_ROOT / 'model'))
sys.path.insert(0, str(REPO_ROOT / 'train'))
from model import DecoderLM
from train import lr_at_step, save_checkpoint, load_checkpoint

# Corpus splits are not in the git repo (data/splits/ is gitignored, Drive-only).
# They live at Drive: individual-project-abhinavborah/<language>/splits/{train,val,test}.txt
DRIVE_ROOT = Path('/content/drive/MyDrive/individual-project-abhinavborah')
DATA_DIR = DRIVE_ROOT / LANGUAGE / 'splits'

CHECKPOINT_DIR = Path('/content/drive/MyDrive/lma_phase2_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f'{LANGUAGE}_checkpoint.pt'
TRAIN_BIN = CHECKPOINT_DIR / f'{LANGUAGE}_train.bin'
VAL_BIN = CHECKPOINT_DIR / f'{LANGUAGE}_val.bin'
TOKENIZER_PATH = REPO_ROOT / 'tokenizer' / 'assamese_bpe_8000.model'

BATCH_SIZE = 32
CHECKPOINT_EVERY = 1000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

In [ ]:
def tokenize_to_memmap(text_path, tokenizer_path, out_path, chunk_chars=20_000_000):
    """Stream-tokenize a large text file to a uint16 binary file on disk.
    Splits on line boundaries, not raw character counts -- cutting a chunk
    mid-line risks a BPE merge that would have spanned the cut point,
    producing different ids than encoding the whole file at once (this was
    verified against direct encoding on a real sample before use here).

    Writes to a temp file first, then renames onto out_path only once
    tokenization finishes. If a session dies mid-tokenize, out_path never
    gets created, so a rerun's exists() check retokenizes cleanly instead
    of silently reusing a truncated file.

    Prints progress after every chunk -- this step can take an hour or
    more against a Drive-mounted output path, and with no output at all
    there is no way to tell a slow run from a hung one."""
    sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
    total = 0
    chars_read = 0
    leftover = ''
    t0 = time.time()
    tmp_path = Path(str(out_path) + '.tmp')
    with open(text_path, encoding='utf-8') as src, open(tmp_path, 'wb') as dst:
        while True:
            chunk = src.read(chunk_chars)
            if not chunk:
                break
            chars_read += len(chunk)
            buf = leftover + chunk
            last_newline = buf.rfind('\n')
            if last_newline == -1:
                leftover = buf
                continue
            to_encode = buf[: last_newline + 1]
            leftover = buf[last_newline + 1 :]
            ids = sp.encode(to_encode, out_type=int)
            np.array(ids, dtype=np.uint16).tofile(dst)
            total += len(ids)
            elapsed = time.time() - t0
            print(f'  ...{chars_read:,} chars read, {total:,} tokens written, {elapsed:.0f}s elapsed')
        if leftover:
            ids = sp.encode(leftover, out_type=int)
            np.array(ids, dtype=np.uint16).tofile(dst)
            total += len(ids)
    os.replace(tmp_path, out_path)
    return total


def load_memmap(bin_path):
    return np.memmap(bin_path, dtype=np.uint16, mode='r')


def get_batch_memmap(data, batch_size, context_length, step):
    """Same deterministic step-indexed batch selection as train.py's
    get_batch, adapted for a numpy memmap backing a corpus too large to
    fit as a single in-memory tensor."""
    span = batch_size * context_length
    max_start = len(data) - context_length - 1
    start = (step * span) % max_start
    xs, ys = [], []
    for b in range(batch_size):
        s = (start + b * context_length) % max_start
        xs.append(np.array(data[s : s + context_length], dtype=np.int64))
        ys.append(np.array(data[s + 1 : s + context_length + 1], dtype=np.int64))
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys))

In [ ]:
with open(REPO_ROOT / 'configs' / 'model_config.json', encoding='utf-8') as f:
    full_config = json.load(f)
model_config = full_config['model']
opt_config = full_config['optimizer']
sched_config = full_config['schedule']
assert sched_config['peak_lr'] is not None, 'run the local smoke test first to pick peak_lr/min_lr'

assert DATA_DIR.exists(), f'corpus splits not found on Drive at {DATA_DIR} -- check the Drive folder layout'

if not TRAIN_BIN.exists():
    print('tokenizing full train split to disk (one-time, persists on Drive)...')
    n = tokenize_to_memmap(DATA_DIR / 'train.txt', TOKENIZER_PATH, TRAIN_BIN)
    print(f'train tokens: {n:,}')
if not VAL_BIN.exists():
    print('tokenizing full val split to disk (one-time, persists on Drive)...')
    n = tokenize_to_memmap(DATA_DIR / 'val.txt', TOKENIZER_PATH, VAL_BIN)
    print(f'val tokens: {n:,}')

train_data = load_memmap(TRAIN_BIN)
total_tokens = len(train_data)
tokens_per_step = BATCH_SIZE * model_config['context_length']
total_steps = total_tokens // tokens_per_step
# ~1% of total steps, floored at the config's own value -- 200 steps of
# warmup suits a short run but is too small a fraction for a ~100k+ step
# full run; scale with total_steps instead of hardcoding.
warmup_steps = max(sched_config['warmup_steps'], total_steps // 100)
print(f'total_steps={total_steps:,} (one epoch), warmup_steps={warmup_steps:,}')

In [ ]:
model = DecoderLM(copy.deepcopy(model_config))
optimizer = torch.optim.AdamW(
    model.parameters(), betas=(opt_config['beta1'], opt_config['beta2']),
    weight_decay=opt_config['weight_decay'],
)
model.to(device)

start_step = 0
if CHECKPOINT_PATH.exists():
    start_step = load_checkpoint(CHECKPOINT_PATH, model, optimizer, map_location=device)
    print(f'resumed from step {start_step:,}')
else:
    print('no existing checkpoint, starting fresh')

In [ ]:
model.train()
t0 = time.time()
for step in range(start_step, total_steps):
    lr = lr_at_step(step, warmup_steps, sched_config['peak_lr'], sched_config['min_lr'], total_steps)
    for group in optimizer.param_groups:
        group['lr'] = lr

    torch.manual_seed(step)
    x, y = get_batch_memmap(train_data, BATCH_SIZE, model_config['context_length'], step)
    x, y = x.to(device), y.to(device)

    logits = model(x)
    loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), opt_config['grad_clip_norm'])
    optimizer.step()

    if step % 100 == 0:
        elapsed = time.time() - t0
        print(f'step {step:,}/{total_steps:,} loss={loss.item():.4f} lr={lr:.2e} elapsed={elapsed:.0f}s')

    if (step + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(CHECKPOINT_PATH, model, optimizer, step + 1, full_config)
        print(f'checkpoint saved at step {step + 1:,}')

save_checkpoint(CHECKPOINT_PATH, model, optimizer, total_steps, full_config)
print('training complete, final checkpoint saved')